# 🚀 BSL Coder Fine-tuning

Fine-tuning Qwen2.5-Coder-7B на коде 1С:Предприятие (BSL)

**Требования:**
- Runtime → Change runtime type → GPU (T4)
- Датасет загружен в Google Drive

**Автор:** Claude Code Framework
**Дата:** 2025-12-08

## 1. Установка зависимостей

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Проверка GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Путь к датасету на Google Drive
# Загрузите bsl_training.json в папку My Drive/BSL_Finetuning/
DATASET_PATH = "/content/drive/MyDrive/BSL_Finetuning/bsl_training.json"
OUTPUT_DIR = "/content/drive/MyDrive/BSL_Finetuning/output"

import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Загрузка модели

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("✅ Модель загружена!")

## 4. Настройка LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ LoRA адаптеры настроены!")

## 5. Загрузка датасета

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_PATH)
print(f"📊 Загружено примеров: {len(dataset['train'])}")

# Промпт-шаблон
alpaca_prompt = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""


def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)
print("✅ Датасет подготовлен!")

## 6. Обучение

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Расчёт шагов
num_examples = len(dataset["train"])
batch_size = 2
gradient_accumulation = 4
effective_batch_size = batch_size * gradient_accumulation
steps_per_epoch = num_examples // effective_batch_size
max_steps = min(steps_per_epoch, 500)  # Макс 500 шагов для первого прогона

print("📈 Настройки обучения:")
print(f"   Примеров: {num_examples}")
print(f"   Effective batch: {effective_batch_size}")
print(f"   Шагов: {max_steps}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        warmup_steps=10,
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=f"{OUTPUT_DIR}/checkpoints",
        save_steps=100,
        save_total_limit=3,
    ),
)

In [ ]:
# 🚀 Запуск обучения
print("🚀 Начинаем обучение...")
trainer_stats = trainer.train()
print("\n✅ Обучение завершено!")
print(f"   Время: {trainer_stats.metrics['train_runtime']:.0f} сек")
print(f"   Loss: {trainer_stats.metrics['train_loss']:.4f}")

## 7. Сохранение модели

In [ ]:
# Сохранение LoRA адаптеров
LORA_PATH = f"{OUTPUT_DIR}/bsl-coder-lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"✅ LoRA сохранены: {LORA_PATH}")

In [ ]:
# Экспорт в GGUF для Ollama
GGUF_PATH = f"{OUTPUT_DIR}/gguf"
os.makedirs(GGUF_PATH, exist_ok=True)

model.save_pretrained_gguf(GGUF_PATH, tokenizer, quantization_method="q4_k_m")
print(f"✅ GGUF экспортирован: {GGUF_PATH}")

## 8. Тестирование модели

In [ ]:
# Включаем режим инференса
FastLanguageModel.for_inference(model)

# Тестовый промпт
test_prompt = alpaca_prompt.format(
    instruction="Напиши функцию для проверки корректности ИНН на языке 1С (BSL)",
    input="",
    output="",
)

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    top_p=0.9,
)

result = tokenizer.batch_decode(outputs)[0]
print("📝 Результат генерации:")
print("=" * 50)
print(result.split("### Response:")[1] if "### Response:" in result else result)

## 9. Скачивание модели

После завершения обучения:
1. Скачайте GGUF файл из Google Drive
2. Поместите в `D:\1C-Enterprise_Framework\data\models\gguf\`
3. Загрузите в Ollama:
```bash
ollama create bsl-coder -f D:\1C-Enterprise_Framework\data\models\Modelfile
```

In [ ]:
# Список файлов для скачивания
print("📁 Файлы для скачивания:")
print(f"   {GGUF_PATH}/")
!ls -la "{GGUF_PATH}"